In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model_name = os.getenv("MODEL_NAME")
model_key=os.getenv("API_KEY")
print(model_key)

sk-PPA3eElT-wlDngGwfPGsdw


In [4]:
import dspy
from openai import OpenAI

In [5]:
kode_model = dspy.LM(
    f"openai/{model_name}", 
    api_key=model_key,
    base_url = "https://api.ai.kodekloud.com/v1"
)

# 2. Configure DSPy to use this model globally
dspy.configure(lm=kode_model )

In [6]:
response=kode_model("Who is the primeminister of canada")
print(response[0])

The current Prime Minister of Canada is **Justin Trudeau**.

If you want, I can also give you the **current date-sensitive answer** based on the latest available information, since political offices can change.


In [19]:
predict =dspy.Predict("question -> answer")
prediction = predict(question="Who schored the final goal in football world cup finals in 2014")
prediction.answer

'Mario Götze scored the final goal in the 2014 FIFA World Cup final.'

In [7]:
print(kode_model.inspect_history(1))





[2026-09-22T19:43:06.575163]

User message:

Who is the primeminister of canada


Response:

The current Prime Minister of Canada is **Justin Trudeau**.

If you want, I can also give you the **current date-sensitive answer** based on the latest available information, since political offices can change.





None


Signature:
A signature is a declarative specification of input/output behavior of a DSPy module. Signatures allow you to tell the LM what it needs to do, rather than speify how we should ask the LM to do it.

In [21]:
class QA(dspy.Signature):
    question = dspy.InputField()
    answer =dspy.OutputField()

predict = dspy.Predict(QA)
prediction = predict(question="Who scored the final goal in football world cup finals in 2014?")
print(prediction.answer)

Mario Götze scored the final goal in the 2014 FIFA World Cup final.


In [8]:
kode_model.inspect_history(n=1)





[2026-09-22T19:43:06.575163]

User message:

Who is the primeminister of canada


Response:

The current Prime Minister of Canada is **Justin Trudeau**.

If you want, I can also give you the **current date-sensitive answer** based on the latest available information, since political offices can change.







In [13]:
class QA(dspy.Signature):
    """Get names from computing. If you don't find tell Not known """
    question = dspy.InputField(desc="User's question")
    answer =dspy.OutputField(desc="often between 2 to 5 lines. Name in all Caps, name appended with the country name")

predict = dspy.Predict(QA)
prediction = predict(question="Who is Charles babbage?")
print(prediction.answer)

CHARLES BABBAGE - UNITED KINGDOM
English mathematician, philosopher, inventor, and mechanical engineer.
Known as the "father of the computer" for his concept of a programmable mechanical computer.


In [14]:
kode_model.num_retries

3

In [15]:
kode_model.history[-1]["messages"]

[{'role': 'system',
  'content': "Your input fields are:\n1. `question` (str): User's question\nYour output fields are:\n1. `answer` (str): often between 2 to 5 lines. Name in all Caps, name appended with the country name\nAll interactions will be structured in the following way, with the appropriate values filled in.\n\n[[ ## question ## ]]\n{question}\n\n[[ ## answer ## ]]\n{answer}\n\n[[ ## completed ## ]]\nIn adhering to this structure, your objective is: \n        Get names from computing. If you don't find tell Not known "},
 {'role': 'user',
  'content': '[[ ## question ## ]]\nWho is Charles babbage?\n\nRespond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.'}]

## Signature -QuestionAnswering

In [ ]:
# Class-based signature
# There is no call to the llm defined in the signature, it is always done outside
class QuestionAnswering(dspy.Signature):
    """Answer questions with short factual responses."""

    question: str = dspy.InputField()
    answer : str = dspy.OutputField(desc="A concise factual answer")

print("Signature defined")
print(f" Input: {QuestionAnswering.fields.keys()}")

Signature defined
 Input: dict_keys(['question', 'answer'])


# Modules

Modules will control how your signatures will execute,

## dspy.Predict - Basic Prediction

In [17]:
# In the modules we do call the llm
predictor = dspy.Predict(QuestionAnswering)
result = predictor(question="What is the capital of canada?")
print("Question: What is the capital of Canada?")
print(f"Answer: {result.answer}")

Question: What is the capital of Canada?
Answer: Ottawa


# dspy.ChainOfThought - Adding Rational (COT)

In [18]:
# ChainOfThought automatically adds a 'reasoning' field. (dspy 3.x renamed 'rationale')

cot=dspy.ChainOfThought(QuestionAnswering)
result = cot(question="If the present age of the Son is 15 years and 5 years back his father was 3 times older than the son, guess the father's present age.")
print("Rationale:")
print(result.reasoning)
print(f"\n Answer: {result.answer}")

Rationale:
The son is now 15. Five years ago, the son was 10. At that time, the father was 3 times the son's age, so the father was 30. Therefore, the father's present age is 35.

 Answer: 35 years
